# Publications markdown generator for academicpages

Takes a TSV of publications with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `publications.py`. Run either from the `markdown_generator` folder after replacing `publications.tsv` with one containing your data.

TODO: Make this work with BibTex and other databases of citations, rather than Stuart's non-standard TSV format and citation style.


## Data format

The TSV needs to have the following columns: pub_date, title, venue, excerpt, citation, site_url, and paper_url, with a header at the top. 

- `excerpt` and `paper_url` can be blank, but the others must have values. 
- `pub_date` must be formatted as YYYY-MM-DD.
- `url_slug` will be the descriptive part of the .md file and the permalink URL for the page about the paper. The .md file will be `YYYY-MM-DD-[url_slug].md` and the permalink will be `https://[yourdomain]/publications/YYYY-MM-DD-[url_slug]`

This is how the raw file looks (it doesn't look pretty, use a spreadsheet or other program to edit and create).

In [1]:
!cat publications.tsv

pub_date	title	venue	excerpt	citation	url_slug	paper_url	slides_url
2020-05-20	Padé Approximants and the Anharmonic Oscillator	Yale University		Desai, Krish. (2020). "Padé Approximants and the Anharmonic Oscillator." <i>Yale University</i>. MS Mathematics Thesis.	pade-approximants-anharmonic-oscillator	https://www.desai.ml/files/2-paper.pdf	
2021-11-01	Symmetry Discovery with Deep Learning	NeurIPS 2021		Desai, K., Nachman, B., & Thaler, J. (2021). Symmetry Discovery with Deep Learning. <i>NeurIPS</i> ML4PS 117 (2021)	symmetry-discovery-deep-learning	https://www.desai.ml/files/3-paper.pdf	https://www.desai.ml/files/3-slides.pdf
2021-12-16	Oblivious points on translation surfaces	Journal of Geometry		Adelstein, I., Desai, K., Ji, A., & Zdeblick, G. (2022). Oblivious points on translation surfaces. <i>Journal of Geometry</i>, 113(1), 6.	oblivious-points-translation-surfaces	https://www.desai.ml/files/1-paper.pdf	https://www.desai.ml/files/1-slides.pdf
2022-05-01	SymmetryGAN	Physical Revie

## Import pandas

We are using the very handy pandas library for dataframes.

In [2]:
import pandas as pd

## Import TSV

Pandas makes this easy with the read_csv function. We are using a TSV, so we specify the separator as a tab, or `\t`.

I found it important to put this data in a tab-separated values format, because there are a lot of commas in this kind of data and comma-separated values can get messed up. However, you can modify the import statement, as pandas also has read_excel(), read_json(), and others.

In [3]:
publications = pd.read_csv("publications.tsv", sep="\t", header=0)
publications


,pub_date,title,venue,excerpt,citation,url_slug,paper_url,slides_url
0,2020-05-20,Padé Approximants and the Anharmonic Oscillator,Yale University,NaN,"Desai, Krish. (2020). ""Padé Approximants and t...",pade-approximants-anharmonic-oscillator,https://www.desai.ml/files/2-paper.pdf,NaN
1,2021-11-01,Symmetry Discovery with Deep Learning,NeurIPS 2021,NaN,"Desai, K., Nachman, B., & Thaler, J. (2021). S...",symmetry-discovery-deep-learning,https://www.desai.ml/files/3-paper.pdf,https://www.desai.ml/files/3-slides.pdf
2,2021-12-16,Oblivious points on translation surfaces,Journal of Geometry,NaN,"Adelstein, I., Desai, K., Ji, A., & Zdeblick, ...",oblivious-points-translation-surfaces,https://www.desai.ml/files/1-paper.pdf,https://www.desai.ml/files/1-slides.pdf
3,2022-05-01,SymmetryGAN,Physical Review D,NaN,"Desai, K., Nachman, B., & Thaler, J. (2022). S...",symmetrygan,https://www.desai.ml/files/4-paper.pdf,NaN
4,2022-11-01,Deconvolving Detector Effects for Distribution...,NeurIPS ML4PS,NaN,"Desai, K., Nachman, B., & Thaler, J. (2022). D...",deconvolving-detector-effects,https://www.desai.ml/files/5-paper.pdf,https://www.desai.ml/files/5-slides.pdf
5,2024-11-01,Multidimensional Deconvolution with Profiling,NeurIPS ML4PS,NaN,"Zhu, H., Desai, K., Kuusela, M., Mikuni, V., N...",multidimensional-deconvolution-profiling,https://www.desai.ml/files/8-paper.pdf,https://www.desai.ml/files/8-slides.pdf
6,2024-11-01,Neural Posterior Unfolding,NeurIPS ML4PS,NaN,"Acosta, F. T., Chan, J., Desai, K., Mikuni, V....",neural-posterior-unfolding,https://www.desai.ml/files/7-paper.pdf,https://www.desai.ml/files/7-slides.pdf
7,2024-12-13,Moment Unfolding,Physical Review D,NaN,"Desai, K., Nachman, B., & Thaler, J. (2024). M...",moment-unfolding,https://www.desai.ml/files/6-paper.pdf,NaN
8,2025-04-18,Unbinned Inference with Correlated Events,arXiv,NaN,"Desai, K., Long, O., & Nachman, B. (2025). Unb...",unbinned-inference-correlated-events,https://www.desai.ml/files/9-paper.pdf,NaN
9,2025-08-15,Machine Learning Methods for Cross Section Mea...,"University of California, Berkeley",NaN,"Desai, Krish. (2025). Machine Learning Methods...",machine-learning-cross-section-measurements,https://www.desai.ml/files/10-paper.pdf,NaN


## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [4]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

In [5]:
import os
for row, item in publications.iterrows():
    
    md_filename = str(item.pub_date) + "-" + item.url_slug + ".md"
    html_filename = str(item.pub_date) + "-" + item.url_slug
    year = item.pub_date[:4]
    
    ## YAML variables
    
    md = "---\ntitle: \""   + item.title + '"\n'
    
    md += """collection: publications"""
    
    md += """\npermalink: /publication/""" + html_filename
    
    if len(str(item.excerpt)) > 5:
        md += "\nexcerpt: '" + html_escape(item.excerpt) + "'"
    
    md += "\ndate: " + str(item.pub_date) 
    
    md += "\nvenue: '" + html_escape(item.venue) + "'"
    
    if len(str(item.slides_url)) > 5:
        md += "\nslidesurl: '" + item.slides_url + "'"

    if len(str(item.paper_url)) > 5:
        md += "\npaperurl: '" + item.paper_url + "'"
    
    md += "\ncitation: '" + html_escape(item.citation) + "'"
    
    md += "\n---"
    
    ## Markdown description for individual page
        
    if len(str(item.excerpt)) > 5:
        md += "\n" + html_escape(item.excerpt) + "\n"

    if len(str(item.slides_url)) > 5:
        md += "\n[Download slides here](" + item.slides_url + ")\n" 

    if len(str(item.paper_url)) > 5:
        md += "\n[Download paper here](" + item.paper_url + ")\n" 
        
    md += "\nRecommended citation: " + item.citation
    
    md_filename = os.path.basename(md_filename)
       
    with open("../_publications/" + md_filename, 'w') as f:
        f.write(md)

These files are in the publications directory, one directory below where we're working from.

In [15]:
!ls ../_publications/

2009-10-01-paper-title-number-1.md  2015-10-01-paper-title-number-3.md
2010-10-01-paper-title-number-2.md  2024-02-17-paper-title-number-4.md


In [16]:
!cat ../_publications/2009-10-01-paper-title-number-1.md

---
title: "Paper Title Number 1"
collection: publications
permalink: /publication/2009-10-01-paper-title-number-1
excerpt: 'This paper is about the number 1. The number 2 is left for future work.'
date: 2009-10-01
venue: 'Journal 1'
slidesurl: 'http://academicpages.github.io/files/slides1.pdf'
paperurl: 'http://academicpages.github.io/files/paper1.pdf'
citation: 'Your Name, You. (2009). &quot;Paper Title Number 1.&quot; <i>Journal 1</i>. 1(1).'
---
This paper is about the number 1. The number 2 is left for future work.

[Download slides here](http://academicpages.github.io/files/slides1.pdf)

[Download paper here](http://academicpages.github.io/files/paper1.pdf)

Recommended citation: Your Name, You. (2009). "Paper Title Number 1." <i>Journal 1</i>. 1(1).